In [3]:
# add this import near the top with the others
import os


In [5]:
!pip install wikipedia


In [ ]:
#libraries
import sys, re, datetime
import wikipedia
import wikipedia.exceptions as wiki_exceptions
import nltk
from nltk.tokenize import sent_tokenize
nltk.download("punkt", quiet=True)
wikipedia.set_lang("en")

language_codes = {
    "en": "en", "english": "en",
    "es": "es", "spanish": "es", "español": "es",
    "fr": "fr", "french": "fr", "français": "fr",
    "de": "de", "german": "de", "deutsch": "de",
    "it": "it", "italian": "it", "italiano": "it",
    "pt": "pt", "portuguese": "pt", "português": "pt",
    "ru": "ru", "russian": "ru", "русский": "ru",
    "zh": "zh-CN", "chinese": "zh-CN", "中文": "zh-CN", "简体中文": "zh-CN",
    "zh-cn": "zh-CN", "zh-tw": "zh-TW", "繁體中文": "zh-TW",
    "ja": "ja", "japanese": "ja", "日本語": "ja",
    "ko": "ko", "korean": "ko", "한국어": "ko",
    "hi": "hi", "hindi": "hi", "हिन्दी": "hi",
    "ar": "ar", "arabic": "ar", "العربية": "ar",
    "tr": "tr", "turkish": "tr", "türkçe": "tr",
    "nl": "nl", "dutch": "nl", "nederlands": "nl",
    "sv": "sv", "swedish": "sv", "svenska": "sv",
    "pl": "pl", "polish": "pl", "polski": "pl",
    "vi": "vi", "vietnamese": "vi", "tiếng việt": "vi",
    "th": "th", "thai": "th", "ไทย": "th",
}

def simplified_text(s: str) -> str:
    if not s: return "en"
    key = s.strip().lower()
    return language_codes.get(key, "en")

def _translate(text: str, src: str = "en", dest: str = "en") -> str:
    return text

# -------------------- Basic text cleaning --------------------
bracket_re = re.compile(r"\[[^\]]*\]")
parenthesis_re = re.compile(r"\([^)]*\)")
space_re = re.compile(r"\s+")

def strip_noise(s: str) -> str:
    s = bracket_re.sub("", s)
    s = parenthesis_re.sub("", s)
    s = space_re.sub(" ", s).strip()
    return s

def sentences(text: str):
    if not text: return []
    try:
        return [t.strip() for t in sent_tokenize(text) if t.strip()]
    except Exception:
        return [t.strip() for t in text.split("\n") if t.strip()]

def first_paragraph(text: str) -> str:
    return " ".join(sentences(text)[:3])

# -------------------- Question parsing --------------------
def question_type(q: str):
    q = q.strip().lower()
    if q.startswith("who"): return "who"
    if q.startswith("what"): return "what"
    if q.startswith("when"): return "when"
    if q.startswith("where"): return "where"
    return None

def extract_entity(question: str):
    q = question.strip().rstrip("?")
    parts = q.split(maxsplit=1)
    if len(parts) == 1: return q.title()
    rest = parts[1]
    rest = re.sub(r"^(is|was|were|did|does|do|are|has|have|had|the|a|an)\b", "", rest, flags=re.I).strip()
    rest = re.sub(r"\b(born|founded|created|established|built|located|situated|held|invented|started|in|on|at)\b.*", "", rest, flags=re.I).strip()
    return (rest or q).title()

# -------------------- Wikipedia fetch --------------------
def searching_wiki(entity: str, k=5):
    out, seen = [], set()
    try:
        titles = wikipedia.search(entity, results=k) or [entity]
    except Exception:
        titles = [entity]
    for t in titles:
        if t in seen: continue
        try:
            page = wikipedia.page(t, auto_suggest=False)
            txt = (page.summary or "") + "\n" + (page.content or "")
            out.append((page.title, txt)); seen.add(t)
        except wikipedia.exceptions.DisambiguationError as de:
            for opt in de.options[:k]:
                try:
                    p = wikipedia.page(opt)
                    txt = (p.summary or "") + "\n" + (p.content or "")
                    if p.title not in seen:
                        out.append((p.title, txt)); seen.add(p.title)
                except Exception:
                    pass
        except Exception:
            try:
                s = wikipedia.summary(t)
                out.append((t, s)); seen.add(t)
            except Exception:
                pass
    return out

# -------------------- WHEN extractor (full sentence) --------------------
when_keywords = r"(?:was|were|is|are|has been|have been|began|started|launched|founded|established|created|opened|signed|occurred|took place|ended|born)"
current_year  = datetime.datetime.utcnow().year
date_detail  = re.compile(r"\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s+(\d{4})\b", re.I)
date_alternative   = re.compile(r"\b(\d{1,2})\s+(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{4})\b", re.I)
numeric_date = re.compile(r"\b(\d{1,2})/(\d{1,2})/(\d{2,4})\b")
year_date  = re.compile(r"\b(1[6-9]\d{2}|20\d{2}|21\d{2})\b")
year_range = re.compile(r"\b\d{4}\s*[–-]\s*\d{4}\b")

def choosing_date(span: str) -> str:
    s = year_range.sub("", span)
    full = [(m.group(0), int(m.group(1))) for m in date_detail.finditer(s)]
    if full:  return min(full, key=lambda x:x[1])[0] + "."
    alt  = [(m.group(0), int(m.group(3))) for m in date_alternative.finditer(s)]
    if alt:   return min(alt, key=lambda x:x[1])[0] + "."
    slash=[]
    for m in numeric_date.finditer(s):
        y = m.group(3); y = int(y) if len(y)==4 else int(f"20{y}")
        slash.append((m.group(0), y))
    if slash:  return min(slash, key=lambda x:x[1])[0] + "."
    years = [int(y) for y in year_date.findall(s) if 1600 <= int(y) <= current_year]
    return (f"{min(years)}." if years else "")

def making_sentences(entity: str, verb: str, date_with_period: str) -> str:
    """Build a clean sentence from an extracted verb + date."""
    date_tail = date_with_period.rstrip(".")
    v = (verb or "").lower()

    if "born" in v:
        return f"{entity} was born on {date_tail}."
    if any(w in v for w in ["founded", "established", "created", "opened", "built", "formed"]):
        base = v.split()[-1]  # 'founded' from 'was founded'
        # prefer 'in <year>' when date is a bare year
        if re.fullmatch(r"\d{4}", date_tail):
            return f"{entity} was {base} in {date_tail}."
        return f"{entity} was {base} on {date_tail}."
    if any(w in v for w in ["began", "started", "launched", "signed", "occurred", "took place", "ended"]):
        prep = "in" if re.fullmatch(r"\d{4}", date_tail) else "on"
        return f"{entity} {v} {prep} {date_tail}."
    # Fallback
    prep = "in" if re.fullmatch(r"\d{4}", date_tail) else "on"
    return f"{entity} was {prep} {date_tail}."

def extract_when(entity: str, sent: str) -> str:
    """
    Find a relevant date near WHEN verbs and return a FULL sentence,
    e.g., '<Entity> was born on Month D, YYYY.' or '<Entity> was founded in 19XX.'
    """
    s = strip_noise(sent)
    if entity.lower() not in s.lower(): return ""
    # Prefer a date near an explicit WHEN verb
    for m in re.finditer(when_keywords, s, re.I):
        verb = m.group(0)
        span = s[m.start(): m.end()+120]
        d = choosing_date(span)
        if d:
            return making_sentences(entity, verb, d)
    # Otherwise, any date in the sentence → still produce a sentence
    d = choosing_date(s)
    if d:
        return making_sentences(entity, "", d)
    return ""

# -------------------- WHERE extractor (location + sentence) --------------------
where_keyword = r"(?:is|was|are|were|stands|lies|sits|located|situated|based)"
place_name  = r"[A-Z][A-Za-z0-9 .'\-]+"
place_template  = rf"{place_name}(?:,\s*{place_name})*"

WHERE_PATTERNS = [
    re.compile(rf"\b{where_keyword}\s+(?:in|at|on|within)\s+({place_template})", re.I),
    re.compile(r",\s+in\s+(" + place_template + r")", re.I),
]

country_name = {
 "United States","India","China","France","Italy","United Kingdom","England","Scotland","Wales",
 "Northern Ireland","Canada","Germany","Japan","Brazil","Russia","Spain","Mexico","Egypt","Turkey",
 "Greece","Nepal","Bhutan","Pakistan","Australia","New Zealand","Peru","Chile","Argentina","Qatar",
 "Saudi Arabia","UAE","United Arab Emirates","Netherlands","Belgium","Switzerland","Austria",
 "Sweden","Norway","Denmark","Finland","Ireland","Portugal","Poland","Czech Republic","Hungary",
 "Romania","Bulgaria","Serbia","Croatia","Slovenia","Slovakia","Ukraine","Belarus","Estonia",
 "Latvia","Lithuania","Iceland","South Africa","Nigeria","Kenya","Ethiopia","Morocco","Algeria",
 "Tunisia","Israel","Iran","Iraq","Syria","Lebanon","Jordan","Singapore","Malaysia","Indonesia",
 "Philippines","Vietnam","Thailand","Cambodia","Laos","Bangladesh","Sri Lanka","South Korea","North Korea"
}
us_states = re.compile(
 r"\b(Alabama|Alaska|Arizona|Arkansas|California|Colorado|Connecticut|Delaware|Florida|Georgia|Hawaii|Idaho|Illinois|Indiana|Iowa|Kansas|Kentucky|Louisiana|Maine|Maryland|Massachusetts|Michigan|Minnesota|Mississippi|Missouri|Montana|Nebraska|Nevada|New Hampshire|New Jersey|New Mexico|New York|North Carolina|North Dakota|Ohio|Oklahoma|Oregon|Pennsylvania|Rhode Island|South Carolina|South Dakota|Tennessee|Texas|Utah|Vermont|Virginia|Washington|West Virginia|Wisconsin|Wyoming)\b$",
 re.I
)
generic_words = {"the world","earth","history","the internet","the web","north","south","east","west"}

place_extras = re.compile(
    r"^(?:the|a|an)\s+|^(?:city|town|state|region|province|district|county|metropolitan area|part)\s+of\s+",
    re.I
)
place_trail  = re.compile(r"(?:;|:|\swhich\s|\sthat\s|\sand\s).*$", re.I)
trail_no   = re.compile(r"\b(?:about|around|approximately)\s+\d[\d.,]*\b.*$", re.I)
has_nos  = re.compile(r"\d")

def location_syntax(p: str) -> str:
    p = place_extras.sub("", p).strip(" ,.")
    if has_nos.search(p): return ""  # drop population/distances/roads
    if p.lower() in generic_words: return ""
    if re.search(r"\b(river|valley|bank|coast|mountain range|plateau|ocean|sea|bay|gulf)\b", p, re.I):
        return ""
    return p

def location_rank(parts):
    score = 0
    if not parts: return -1
    score += min(3, len(parts)) * 2
    last = parts[-1]
    if last in country_name: score += 5
    if us_states.search(last): score += 3
    if 2 <= len(parts) <= 3: score += 4
    if any(len(p) > 40 for p in parts): score -= 3
    return score

def normalizing_location(raw: str, entity: str) -> str:
    raw = strip_noise(raw)
    raw = trail_no.sub("", raw)
    raw = place_trail.sub("", raw)
    if " in " in raw.lower():
        raw = re.split(r"\bin\b", raw, flags=re.I)[-1].strip(" ,.")
    parts = [location_syntax(p) for p in raw.split(",")]
    parts = [p for p in parts if p]
    if not parts: return ""
    if ", ".join(parts).lower() == entity.lower(): return ""
    if us_states.search(parts[-1]) and "United States" not in parts:
        parts.append("United States")
    dedup = []
    for p in parts:
        if not dedup or dedup[-1] != p:
            dedup.append(p)
    parts = dedup[:3]
    if any(p.lower() in generic_words for p in parts):
        return ""
    return ", ".join(parts).rstrip(".") + "."

def extract_where(entity: str, sent: str):
    s = strip_noise(sent)
    if entity.lower() not in s.lower():
        return ""
    cands = []
    for rgx in WHERE_PATTERNS:
        for m in rgx.finditer(s):
            loc = normalizing_location(m.group(1), entity)
            if loc:
                parts = [p.strip() for p in loc.rstrip(".").split(",")]
                cands.append((loc, location_rank(parts)))
    if not cands:
        return ""
    best = max(cands, key=lambda x: x[1])[0]
    if best.strip(".").lower() == entity.lower():
        return ""
    return best  # location WITH final period

# -------------------- WHO/WHAT clause --------------------
def summarize_answer(sentence: str, entity: str) -> str:
    s = strip_noise(sentence)
    m = re.search(rf"\b{re.escape(entity)}\b[^.]*?\b(is|was|are|were)\b\s+(.*)", s, re.I)
    if not m: return ""
    be, rest = m.group(1).lower(), m.group(2)
    cut = re.split(r"(?:\.\s|;|:|\swhich\s|\sthat\s|\sand\s)", rest, maxsplit=1)[0]
    cut = strip_noise(cut).rstrip(" ,;:")
    return f"{entity} {be} {cut}." if cut else ""

# -------------------- Answer search --------------------
SPECIAL_WHERE = {}  # quick overrides, e.g., {"Taj Mahal": "Agra, Uttar Pradesh, India."}

def find_answer(qtype: str, entity: str, candidates):
    # Quick dictionary for where
    if qtype == "where":
        canon = entity.strip().title()
        if canon in SPECIAL_WHERE:
            return f"{entity} is in {SPECIAL_WHERE[canon].rstrip('.') }."

    # Pass 1: scan lead
    for _, text in candidates:
        lead = first_paragraph(text)
        sents = sentences(lead)
        if qtype == "when":
            for s in sents:
                d = extract_when(entity, s)
                if d: return d  # already a sentence
        elif qtype == "where":
            for s in sents:
                loc = extract_where(entity, s)
                if loc: return f"{entity} is in {loc.rstrip('.') }."
        else:
            for s in sents:
                c = summarize_answer(s, entity)
                if c: return c

    # Pass 2: scan more sentences
    for _, text in candidates:
        sents = sentences(text)[:40]
        if qtype == "when":
            for s in sents:
                d = extract_when(entity, s)
                if d: return d
        elif qtype == "where":
            for s in sents:
                loc = extract_where(entity, s)
                if loc: return f"{entity} is in {loc.rstrip('.') }."
        else:
            for s in sents:
                c = summarize_answer(s, entity)
                if c: return c

    return ""

# -------------------- Multilingual I/O loop --------------------
def main():
    # Ask for I/O language once
    name = input("Enter Your Name: ")
    lang_in = input(f"Choose Your preferable language {name} to continue [default is English]: ").strip()
    print(f"The language choosen is {lang_in}")
    target_lang = simplified_text(lang_in)
    src_lang = target_lang  # users type questions in this language

    # Internal Wikipedia/search stays English
    wikipedia.set_lang("en")

    while True:
        try:
            q_native = input("=?> ").strip()
        except EOFError:
            break
        if q_native.lower() == "exit":
            break

        # Translate question to English for parsing (stubbed as no-op)
        q_en = _translate(q_native, src=src_lang, dest="en").strip()

        # Detect qtype on English
        qtype = question_type(q_en)
        if not qtype:
            msg = "Ask a Who/What/When/Where question."
            print(_translate(msg, src="en", dest=target_lang))
            continue

        # Extract entity from English
        entity = extract_entity(q_en)

        # WHERE quick dictionary
        if qtype == "where":
            canon = entity.strip().title()
            if canon in SPECIAL_WHERE:
                ans_en = f"{entity} is in {SPECIAL_WHERE[canon].rstrip('.') }."
                print(_translate(ans_en, src="en", dest=target_lang))
                continue

        # Wikipedia fetch + extract (English)
        cands = searching_wiki(entity, k=5)
        ans_en = find_answer(qtype, entity, cands)

        # Minimal fallback via summary if needed
        if not ans_en:
            try:
                summ = wikipedia.summary(entity, sentences=2)
                if qtype == "when":
                    for s in sentences(summ):
                        d = extract_when(entity, s)
                        if d: ans_en = d; break
                elif qtype == "where":
                    for s in sentences(summ):
                        loc = extract_where(entity, s)
                        if loc: ans_en = f"{entity} is in {loc.rstrip('.') }."; break
                else:
                    for s in sentences(summ):
                        c = summarize_answer(s, entity)
                        if c: ans_en = c; break
            except Exception:
                pass

        # If still nothing: say "I don't know." in chosen language
        if not ans_en:
            print(_translate("I don't know.", src="en", dest=target_lang))
        else:
            # Return single-line answer in chosen language (stubbed)
            print(_translate(ans_en, src="en", dest=target_lang))

if __name__ == "__main__":
    main()


/var/folders/8q/k_54hxvx05x5ml7_73sk06wm0000gn/T/ipykernel_94912/1656728076.py:119: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  _NOW_YEAR  = datetime.datetime.utcnow().year


The language choosen is 
I don't know.
George Washington was an American agricultural scientist.


/opt/anaconda3/lib/python3.12/site-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /opt/anaconda3/lib/python3.12/site-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


Bicycle is a human-powered or motor-assisted, pedal-driven, single-track vehicle, with two wheels attached to a frame, one behind the other..
I don't know.
Taj Mahal is in city of Agra.
I don't know.
George Washington was on February 22, 1732.


final code above

In [4]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.4 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993227 sha256=da15f2b225fbe1d3f784784b5cf74ffff1dd0e71c06e4aa8feb678dc9408c9e3
  Stored in directory: /Users/vivekkumar/Library/Caches/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [10]:
!pip install deep-translator


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.0 MB/s eta 0:00:00
